# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [8]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [9]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [10]:
# Load environment variables used by the embedding function.
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY is required to create embeddings.")

### VectorDB Instance

In [11]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [12]:
# Use the same embedding function when indexing and querying the collection.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY
)

In [13]:
# Reuse the collection when the notebook is run more than once.
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

### Add documents

In [14]:
# Load every JSON game and upsert it so indexing is safe to rerun.
data_dir = "games"
ids = []
documents = []
metadatas = []

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as file:
        game = json.load(file)

    ids.append(os.path.splitext(file_name)[0])
    documents.append(
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
    )
    metadatas.append(game)

collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
)
print(f"Indexed {len(ids)} game records")
print(f"Collection count: {collection.count()}")

Indexed 30 game records
Collection count: 30


### Semantic Search

The query below searches the embedded game descriptions semantically. Chroma ranks the closest matches by distance; lower distance means a closer match.

In [15]:
def search_games(query: str, n_results: int = 5) -> list[dict]:
    """Return the games most semantically similar to a natural-language query."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]
    return [
        {
            "document": document,
            "metadata": metadata,
            "distance": distance,
        }
        for document, metadata, distance in zip(documents, metadatas, distances)
    ]


query = "Which PC games are role-playing games?"
search_results = search_games(query)

print(f"Query: {query}")
print(f"Matches found: {len(search_results)}\n")
for rank, result in enumerate(search_results, start=1):
    metadata = result["metadata"]
    print(f"{rank}. {metadata['Name']} ({metadata['Platform']}, {metadata['YearOfRelease']})")
    print(f"   Distance: {result['distance']:.4f}")
    print(f"   Description: {metadata['Description']}")
    print()

Query: Which PC games are role-playing games?
Matches found: 5

1. The Elder Scrolls V: Skyrim (PC, 2011)
   Distance: 0.1764
   Description: An open-world fantasy role-playing game where players explore Skyrim, develop their character, and battle dragons.

2. The Witcher 3: Wild Hunt (PC, 2015)
   Distance: 0.1849
   Description: An open-world fantasy role-playing game following monster hunter Geralt of Rivia on a quest across a war-torn continent.

3. Pokémon Red and Blue (Game Boy, 1996)
   Distance: 0.2008
   Description: The original Pokémon role-playing games, where players explore the Kanto region, collect Pokémon, and challenge the Pokémon League.

4. Half-Life 2 (PC, 2004)
   Distance: 0.2075
   Description: A story-driven first-person shooter following Gordon Freeman as he fights an alien occupation using innovative physics-based gameplay.

5. The Sims (PC, 2000)
   Distance: 0.2216
   Description: A life simulation game where players create households, build homes, and guide